# AI & Machine Learning — Task 4
## Classification Models, Evaluation Metrics & Handling Imbalanced Data
**Maincrafts Technology** | www.maincrafts.com | hr@maincrafts.com

---
### Objective
In Task-3, we mastered regression models. In Task-4 we **move to classification** — one of the most critical ML problem types in industry.

We will:
- Train and evaluate classification models
- Use proper evaluation metrics beyond accuracy
- Handle class imbalance
- Compare multiple classifiers scientifically


## Step 1 — Import Required Libraries


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score, accuracy_score,
    precision_recall_curve
)

plt.rcParams.update({
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor':   '#FFFFFF',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

BLUE   = '#2563EB'
GREEN  = '#16A34A'
RED    = '#DC2626'
PURPLE = '#7C3AED'
ORANGE = '#D97706'

print('All libraries imported successfully!')

## Step 2 — Load Dataset
Using the **Breast Cancer dataset** from scikit-learn (built-in, no download needed).

| Class | Label | Meaning |
|-------|-------|---------|
| 0 | Malignant | Cancerous |
| 1 | Benign    | Non-cancerous |


In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

n_mal = int((y==0).sum())
n_ben = int((y==1).sum())
ratio = n_ben / n_mal

print(f'Dataset shape: {X.shape}')
print(f'Classes: 0=Malignant ({n_mal}), 1=Benign ({n_ben})')
print(f'Class ratio: {ratio:.2f}:1  (mild imbalance)')
X.head(3)

## Step 3 — Train-Test Split
We use `stratify=y` to preserve the class distribution in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

n_test_mal = int((y_test==0).sum())
n_test_ben = int((y_test==1).sum())
print(f'Train set: {X_train.shape[0]} samples')
print(f'Test  set: {X_test.shape[0]} samples')
print(f'Test class distribution: Malignant={n_test_mal}, Benign={n_test_ben}')

## Step 4 — Feature Scaling
`StandardScaler` normalises features to mean=0, std=1. Essential for Logistic Regression.


In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

mean_val = X_train_sc[:,0].mean()
std_val  = X_train_sc[:,0].std()
print('StandardScaler applied.')
print(f'Mean of first feature (train): {mean_val:.6f}  (should be ~0)')
print(f'Std  of first feature (train): {std_val:.6f}  (should be ~1)')

## Step 5 — Train Baseline Logistic Regression
Logistic Regression is the go-to baseline for binary classification.


In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr  = lr.predict(X_test_sc)
y_prob_lr  = lr.predict_proba(X_test_sc)[:, 1]

acc_lr   = accuracy_score(y_test, y_pred_lr)
prec_lr  = precision_score(y_test, y_pred_lr)
rec_lr   = recall_score(y_test, y_pred_lr)
f1_lr    = f1_score(y_test, y_pred_lr)
auc_lr   = roc_auc_score(y_test, y_prob_lr)

print(f'Accuracy : {acc_lr:.4f}')
print(f'Precision: {prec_lr:.4f}')
print(f'Recall   : {rec_lr:.4f}')
print(f'F1-Score : {f1_lr:.4f}')
print(f'AUC      : {auc_lr:.4f}')

## Step 6 — Confusion Matrix & Classification Report

### Interpretation Guide
| Term | Description |
|------|-------------|
| **True Positive (TP)** | Correctly predicted Benign |
| **True Negative (TN)** | Correctly predicted Malignant |
| **False Positive (FP)** | Malignant predicted as Benign (missed cancer!) |
| **False Negative (FN)** | Benign predicted as Malignant (false alarm) |


In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Heatmap
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant','Benign'],
            yticklabels=['Malignant','Benign'],
            linewidths=2, linecolor='white', ax=axes[0],
            annot_kws={'size':16, 'weight':'bold'})
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].set_title('Confusion Matrix — LR Baseline', fontsize=12, fontweight='bold')

# Metrics bar
metrics = [acc_lr, prec_lr, rec_lr, f1_lr, auc_lr]
labels  = ['Accuracy','Precision','Recall','F1','AUC']
colors  = [BLUE, '#0891B2', GREEN, PURPLE, ORANGE]
bars = axes[1].bar(labels, metrics, color=colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, metrics):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylim(0, 1.12)
axes[1].set_title('LR Baseline — All Metrics', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Score')
plt.tight_layout()
plt.savefig(r'd:\Maincraft Internship\Task 4\nb_plot_cm_lr.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Malignant','Benign']))

## Step 7 — Precision, Recall & F1-Score Interpretation

**Key questions:**
1. **Which metric is more important for medical diagnosis?** → **Recall**
   - Missing a cancer (FN) is far worse than a false alarm (FP)
   - High Recall = we catch as many real cancer cases as possible

2. **What happens if Recall is low?** → Patients with cancer go undetected

3. **Why F1-Score for imbalanced data?** → It balances precision and recall; accuracy alone inflates when one class dominates

**Formulas:**
$$\text{Precision} = \frac{TP}{TP+FP} \qquad \text{Recall} = \frac{TP}{TP+FN} \qquad \text{F1} = \frac{2 \times P \times R}{P+R}$$


## Step 8 — ROC Curve & AUC Score


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
ax.plot(fpr_lr, tpr_lr, color=BLUE, lw=2.5,
        label=f'LR Baseline  (AUC = {auc_lr:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1.2, alpha=0.5, label='Random Chance')
ax.fill_between(fpr_lr, tpr_lr, alpha=0.08, color=BLUE)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — LR Baseline', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
plt.savefig(r'd:\Maincraft Internship\Task 4\nb_plot_roc_lr.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'AUC Score: {auc_lr:.4f}')
print('AUC = 1.0 -> perfect classifier | AUC = 0.5 -> random classifier')

## Step 9 — Handle Class Imbalance
### Technique: `class_weight='balanced'`

The weight for each class is computed as:
$$w_{class} = \frac{n_{samples}}{n_{classes} \times n_{class\_samples}}$$

This penalises errors on the minority class (Malignant) more heavily during training.


In [ ]:
lr_bal = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_bal.fit(X_train_sc, y_train)
y_pred_bal = lr_bal.predict(X_test_sc)
y_prob_bal = lr_bal.predict_proba(X_test_sc)[:, 1]

acc_bal  = accuracy_score(y_test, y_pred_bal)
prec_bal = precision_score(y_test, y_pred_bal)
rec_bal  = recall_score(y_test, y_pred_bal)
f1_bal   = f1_score(y_test, y_pred_bal)
auc_bal  = roc_auc_score(y_test, y_prob_bal)

print('Results — LR Balanced vs LR Baseline:')
header = f"{'Metric':<12} {'Baseline':>10} {'Balanced':>10} {'Change':>10}"
print(header)
print('-' * 45)
for name, v1, v2 in zip(['Accuracy','Precision','Recall','F1','AUC'],
                         [acc_lr,prec_lr,rec_lr,f1_lr,auc_lr],
                         [acc_bal,prec_bal,rec_bal,f1_bal,auc_bal]):
    delta = v2 - v1
    arrow = 'UP' if delta > 0 else 'DN'
    print(f'{name:<12} {v1:>10.4f} {v2:>10.4f} {arrow} {abs(delta):.4f}')

cm_bal = confusion_matrix(y_test, y_pred_bal)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(cm_bal, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Malignant','Benign'],
            yticklabels=['Malignant','Benign'],
            linewidths=2, linecolor='white', ax=axes[0],
            annot_kws={'size':16, 'weight':'bold'})
axes[0].set_title('Confusion Matrix — LR Balanced', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label'); axes[0].set_ylabel('True Label')

fpr_bal, tpr_bal, _ = roc_curve(y_test, y_prob_bal)
axes[1].plot(fpr_lr,  tpr_lr,  color=BLUE,   lw=2, label=f'LR Baseline (AUC={auc_lr:.3f})')
axes[1].plot(fpr_bal, tpr_bal, color=PURPLE, lw=2, linestyle='--', label=f'LR Balanced (AUC={auc_bal:.3f})')
axes[1].plot([0,1],[0,1],'k--',lw=1.2,alpha=0.5)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC: Baseline vs Balanced', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig(r'd:\Maincraft Internship\Task 4\nb_plot_balanced.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nClassification Report (LR Balanced):')
print(classification_report(y_test, y_pred_bal, target_names=['Malignant','Benign']))

## Step 10 — Compare with Decision Tree Classifier
**Decision Tree** does NOT require feature scaling.

Intern must compare:
- Logistic Regression vs Decision Tree
- Stability vs interpretability
- Overfitting behavior


In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)
y_prob_tree = tree.predict_proba(X_test)[:, 1]

acc_tree  = accuracy_score(y_test, y_pred_tree)
prec_tree = precision_score(y_test, y_pred_tree)
rec_tree  = recall_score(y_test, y_pred_tree)
f1_tree   = f1_score(y_test, y_pred_tree)
auc_tree  = roc_auc_score(y_test, y_prob_tree)

# Overfitting check
train_acc_tree = accuracy_score(y_train, tree.predict(X_train))
train_acc_bal  = accuracy_score(y_train, lr_bal.predict(X_train_sc))
print(f'Decision Tree  — Train: {train_acc_tree:.4f}  Test: {acc_tree:.4f}  Gap: {train_acc_tree-acc_tree:.4f}')
print(f'LR Balanced    — Train: {train_acc_bal:.4f}   Test: {acc_bal:.4f}  Gap: {train_acc_bal-acc_bal:.4f}')

cm_tree = confusion_matrix(y_test, y_pred_tree)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(cm_tree, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Malignant','Benign'],
            yticklabels=['Malignant','Benign'],
            linewidths=2, linecolor='white', ax=axes[0],
            annot_kws={'size':16, 'weight':'bold'})
axes[0].set_title('Confusion Matrix — Decision Tree', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label'); axes[0].set_ylabel('True Label')

fpr_tree, tpr_tree, _ = roc_curve(y_test, y_prob_tree)
axes[1].plot(fpr_lr,   tpr_lr,   color=BLUE,   lw=2, label=f'LR Baseline (AUC={auc_lr:.3f})')
axes[1].plot(fpr_bal,  tpr_bal,  color=PURPLE, lw=2, linestyle='--', label=f'LR Balanced (AUC={auc_bal:.3f})')
axes[1].plot(fpr_tree, tpr_tree, color=ORANGE, lw=2, linestyle='-.', label=f'Decision Tree (AUC={auc_tree:.3f})')
axes[1].plot([0,1],[0,1],'k--',lw=1.2,alpha=0.5)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC — All Three Models', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig(r'd:\Maincraft Internship\Task 4\nb_plot_tree_compare.png', dpi=150, bbox_inches='tight')
plt.show()
print('Classification Report (Decision Tree):')
print(classification_report(y_test, y_pred_tree, target_names=['Malignant','Benign']))

## Full Model Comparison Table


In [ ]:
metrics_df = pd.DataFrame({
    'Model':     ['LR Baseline', 'LR Balanced (Best)', 'Decision Tree'],
    'Accuracy':  [acc_lr,  acc_bal,  acc_tree],
    'Precision': [prec_lr, prec_bal, prec_tree],
    'Recall':    [rec_lr,  rec_bal,  rec_tree],
    'F1-Score':  [f1_lr,   f1_bal,   f1_tree],
    'AUC':       [auc_lr,  auc_bal,  auc_tree],
})
metrics_df.set_index('Model', inplace=True)
metrics_df = metrics_df.round(4)
print(metrics_df.to_string())

fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(metrics_df.columns))
w = 0.26
bar_colors = [BLUE, PURPLE, ORANGE]
for i, (model, row) in enumerate(metrics_df.iterrows()):
    offset = (i-1)*w
    bars = ax.bar([xi + offset for xi in x], row.values, w,
                  label=model, color=bar_colors[i], alpha=0.88, edgecolor='white')
    for bar, v in zip(bars, row.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels(metrics_df.columns, fontsize=11)
ax.set_ylim(0, 1.13)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Complete Model Comparison — All Metrics', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(r'd:\Maincraft Internship\Task 4\nb_plot_full_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary & Final Model Decision

### Why NOT just use Accuracy?
- In medical diagnosis, **missing a cancer (False Negative) is far more costly** than a false alarm.
- Accuracy is misleading when classes are imbalanced.
- A model predicting 'Benign' for everyone would get ~63% accuracy but catch **zero cancers**.

### Selected Model: Logistic Regression with `class_weight='balanced'`

| Reason | Detail |
|--------|--------|
| Highest AUC | Best discrimination across all thresholds |
| High Recall | Minimises missed cancer cases |
| Imbalance handled | Auto-adjusts class weights |
| No overfitting | Stable generalisation |
| Interpretable | Coefficient = feature importance |


In [ ]:
joblib.dump(lr_bal, r'd:\Maincraft Internship\Task 4\task4_best_model.pkl')
print('Best model saved: task4_best_model.pkl')
print()
print('=' * 55)
print('  TASK 4 COMPLETE — Final Results')
print('=' * 55)
print(f'  Best Model : Logistic Regression (balanced)')
print(f'  Accuracy   : {acc_bal:.4f}')
print(f'  Precision  : {prec_bal:.4f}')
print(f'  Recall     : {rec_bal:.4f}')
print(f'  F1-Score   : {f1_bal:.4f}')
print(f'  AUC        : {auc_bal:.4f}')
print('=' * 55)